# OptiML — Jay's LightGBM baseline

**First instructions:** Open this file with Colab's **File → Upload notebook**. Choose
**Runtime → Change runtime type → Python 3, CPU / no accelerator**. Run the setup cells
in order. Keep this notebook in your own Drive.

1. Upload `dist/optiml_colab_source.zip` when the first cell asks (rebuild locally with
   `python tools/package_colab.py` after code edits). This route works without pushing Git.
2. Install packages. Restart the session if requested, then resume at configuration.
3. Put all seven CSVs in `MyDrive/OptimlData`, mount Drive and copy them to runtime storage.
4. Prepare features, run the smoke experiment, then run the full baseline.
5. Turn on `FINALIZE` only for a chosen full-data run to generate `submission.csv`.

Use `main` and `jay_patch1` only. Your teammate runs the same shared pipeline locally
with `--model catboost --device GPU`; see README. Save separate run folders.
No credentials or raw data belong in this notebook or Git.

In [ ]:
from pathlib import Path
import os, subprocess, sys, zipfile

SOURCE_MODE = "upload"  # Or "github" after jay_patch1 is pushed.
REPO = Path("/content/Optiml")
if REPO.exists():
    print("Existing source retained:", REPO)
else:
    if SOURCE_MODE == "upload":
        from google.colab import files
        print("Select dist/optiml_colab_source.zip from your computer")
        uploaded = files.upload()
        names = [name for name in uploaded if name.endswith(".zip")]
        if len(names) != 1:
            raise ValueError("Upload exactly one source ZIP")
        REPO.mkdir(parents=True)
        with zipfile.ZipFile(names[0]) as archive:
            for item in archive.infolist():
                destination = (REPO / item.filename).resolve()
                if not destination.is_relative_to(REPO.resolve()):
                    raise ValueError("Unsafe ZIP path")
            archive.extractall(REPO)
    elif SOURCE_MODE == "github":
        subprocess.run(["git", "clone", "--branch", "jay_patch1",
                        "https://github.com/Jaykvankar/Optiml.git", str(REPO)], check=True)
    else:
        raise ValueError("SOURCE_MODE must be upload or github")
os.chdir(REPO)
assert (REPO / "optiml/train.py").exists(), "Upload the generated source archive"
print("Source ready:", REPO)

Select dist/optiml_colab_source.zip from your computer


Saving optiml_colab_source.zip to optiml_colab_source.zip
Source ready: /content/Optiml


In [ ]:
%pip install -q -r /content/Optiml/requirements.txt

## Configure paths and persistent storage
The next cell is safe to run after a session restart. Change DRIVE_DATA if necessary.
Runtime storage is temporary. Finished runs are copied to your Drive directory.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, subprocess, shutil, json, uuid
from google.colab import drive

REPO = Path("/content/Optiml")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
drive.mount("/content/drive")
DRIVE_DATA = Path("/content/drive/MyDrive/OptimlData")
DATA = Path("/content/optiml_data")
PREPARED = Path("/content/optiml_prepared/baseline-v1")
OWNER = "jay"
DRIVE_RUNS = Path("/content/drive/MyDrive/OptimlRuns") / OWNER
DATA.mkdir(parents=True, exist_ok=True)
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)
NAMES = ["train", "test", "customers", "products", "stores", "calendar", "sample_submission"]
for name in NAMES:
    source = DRIVE_DATA / f"{name}.csv"
    if not source.is_file():
        raise FileNotFoundError(f"Missing {source}; place all seven original CSVs in DRIVE_DATA")
    shutil.copy2(source, DATA / source.name)
print("Local CSVs ready:", DATA)

Mounted at /content/drive
Local CSVs ready: /content/optiml_data


## Prepare the shared feature table
This reads all original rows, validates joins and creates features with no learned target
statistics. The training command computes histories inside each split. A completed cache
is reused only when its raw-data hashes and feature-source fingerprint match.

In [ ]:
from optiml.prepare import sha256

feature_source_hash = sha256(REPO / "optiml/features.py")
cache_manifest = PREPARED / "manifest.json"
cache_code = PREPARED / "feature_source.sha256"
if cache_manifest.exists():
    manifest = json.loads(cache_manifest.read_text())
    current = {name: sha256(DATA / f"{name}.csv") for name in NAMES}
    if manifest["source_sha256"] != current or not cache_code.exists() or cache_code.read_text() != feature_source_hash:
        raise ValueError("Data/features changed: choose a new PREPARED path and rerun preparation")
    print("Using matching completed cache:", PREPARED)
else:
    subprocess.run([sys.executable, "-m", "optiml.prepare", "--data-dir", str(DATA),
                    "--output-dir", str(PREPARED)], cwd=REPO, check=True)
    cache_code.write_text(feature_source_hash)

## Run helper
Every call gets a unique directory. Logs stream below. Artifacts are backed up in `finally`,
including partial results on an ordinary Python exception. A hard VM termination can still
lose unsaved local files; the presence of COMPLETE.json distinguishes successful runs.
The default model uses CPU, excludes promotion and includes historical target rates.

In [ ]:
def run_experiment(label, extra_args):
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + label + "_" + uuid.uuid4().hex[:8]
    local = Path("/content/optiml_runs") / OWNER / run_id
    backup = DRIVE_RUNS / run_id
    command = [sys.executable, "-m", "optiml.train", "--prepared-dir", str(PREPARED),
               "--output-dir", str(local), "--model", "lightgbm", "--device", "CPU",
               "--threads", "2", *extra_args]
    print("Running:", " ".join(command), flush=True)
    try:
        subprocess.run(command, cwd=REPO, check=True)
    finally:
        if local.exists():
            shutil.copytree(local, backup, dirs_exist_ok=True)
            print("Backup:", backup)
    return local

## First run: smoke test
This samples complete orders from BOTH periods. It verifies execution and saving, not
competitive AUC. Do not compare this score with full-validation experiments.

In [ ]:
smoke_run = run_experiment("smoke", ["--iterations", "30", "--patience", "10",
                                     "--sample-order-fraction", "0.03"])
print((smoke_run / "metrics.json").read_text())

Running: /usr/bin/python3 -m optiml.train --prepared-dir /content/optiml_prepared/baseline-v1 --output-dir /content/optiml_runs/jay/20261013T122939Z_smoke_255c7c1f --model lightgbm --device CPU --threads 2 --iterations 30 --patience 10 --sample-order-fraction 0.03
Backup: /content/drive/MyDrive/OptimlRuns/jay/20261013T122939Z_smoke_255c7c1f
{
  "auc": 0.5954454960782598,
  "log_loss": 0.47264761169211317,
  "fit_rows": 31352,
  "validation_rows": 9872,
  "fit_positive_rate": 0.184071191630518,
  "validation_positive_rate": 0.1867909238249595,
  "fit_ids_sha256": "65f92e190ed263a15dbf20a678c956af74451a5a71d1c9d8f087c1dec2dd8462",
  "validation_ids_sha256": "ac62a137a2bd552f044cc29e9b7eba9480b8676f17d4afdbf7be41df41ccb0a4",
  "monthly_auc": {
    "2024-07": {
      "rows": 1685,
      "positives": 359,
      "auc": 0.5922633677426402
    },
    "2024-08": {
      "rows": 1325,
      "positives": 268,
      "auc": 0.5830991683022918
    },
    "2024-09": {
      "rows": 1388,
      "posit

## Full LightGBM experiment
For your first full run leave the defaults below. For ablations change one switch at a time.
The October cutoff overlaps the July validation period. Use distinct runs and check the
validation-ID hashes before comparing. Both teammates should use these same split rules.
`FINALIZE=True` runs validation then fits the selected tree count on all labeled data (or
the selected recent window), producing the submission in sample-submission order.

In [ ]:
EXPERIMENT = "lgb_no_promo_history"
CUTOFF = "2024-07-01"  # Recent check: "2024-10-01"
INCLUDE_PROMO = False
USE_HISTORY = True
TRAIN_MONTHS = None   # Alternative: 12 or 6
ITERATIONS = 2000
FINALIZE = False

options = ["--cutoff", CUTOFF, "--iterations", str(ITERATIONS)]
if INCLUDE_PROMO:
    options.append("--include-promo")
if not USE_HISTORY:
    options.append("--no-history")
if TRAIN_MONTHS is not None:
    options += ["--train-months", str(TRAIN_MONTHS)]
if FINALIZE:
    options.append("--finalize")
full_run = run_experiment(EXPERIMENT, options)

Running: /usr/bin/python3 -m optiml.train --prepared-dir /content/optiml_prepared/baseline-v1 --output-dir /content/optiml_runs/jay/20261013T123016Z_lgb_no_promo_history_9f5bd601 --model lightgbm --device CPU --threads 2 --cutoff 2024-07-01 --iterations 2000
Backup: /content/drive/MyDrive/OptimlRuns/jay/20261013T123016Z_lgb_no_promo_history_9f5bd601


In [ ]:
import pandas as pd

metrics = json.loads((full_run / "metrics.json").read_text())
print("Validation AUC:", metrics["auc"])
print("Best iterations:", metrics["best_iterations"])
print("Validation ID hash:", metrics["validation_ids_sha256"])
display(pd.DataFrame(metrics["monthly_auc"]).T)
display(pd.read_csv(full_run / "feature_importance.csv").head(20))
if (full_run / "submission.csv").exists():
    submission = pd.read_csv(full_run / "submission.csv")
    assert len(submission) == 479995
    assert submission.order_line_id.is_unique
    assert submission.returned.notna().all() and submission.returned.between(0, 1).all()
    display(submission.head())
    print("Submission already backed up to Drive. Download from the Files panel if desired.")

Validation AUC: 0.6864069862149578
Best iterations: 137
Validation ID hash: a7db43e1a1fecd83be16f9370da3cc72b5d2639864c91ecce5f512a3e7d8e711


,rows,positives,auc
2024-07,59589.0,10757.0,0.694892
2024-08,58214.0,10560.0,0.688886
2024-09,55090.0,9937.0,0.669865
2024-10,61345.0,11256.0,0.681042
2024-11,58821.0,10517.0,0.691641
2024-12,58850.0,10698.0,0.691917
2025-01,14.0,2.0,0.666667


,feature,importance
0,customer_id,182568.912392
1,product_id,169972.936974
2,product_supplier_code,128029.553452
3,product_id_past_rate,119169.329544
4,store_id,108289.567680
5,customer_store_region_match,107337.896744
6,customer_customer_type,36171.017586
7,store_id_past_rate,35418.513840
8,customer_id_past_rate,34252.654907
9,discount,28137.947151


## What to share with your teammate
Share metrics, config, fold manifest and validation predictions from each useful run.
Use identical validation IDs and join predictions by `order_line_id`. Your teammate can
run `python -m optiml.train --prepared-dir prepared/baseline-v1 --output-dir runs/teammate/cb01 --model catboost --device GPU` locally
after installing `requirements-catboost.txt` and running the same prepare command.
Keep the branch list as `main` and `jay_patch1`; experiment folders do not need Git branches.

References: [LightGBM early stopping](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.early_stopping.html),
[Colab runtime FAQ](https://research.google.com/colaboratory/faq.html).

In [ ]:
import json
import pandas as pd

baseline_run = full_run
baseline_metrics = json.loads(
    (baseline_run / "metrics.json").read_text()
)

print("Baseline:", baseline_run)
display(pd.DataFrame(baseline_metrics["cohorts"]).T)

Baseline: /content/optiml_runs/jay/20261013T123016Z_lgb_no_promo_history_9f5bd601


,rows,positives,auc
channel_catalog,24801.0,4660.0,0.616485
channel_marketplace,109469.0,23768.0,0.666273
channel_online,215805.0,34964.0,0.696356
channel_store,1848.0,335.0,0.614010
unseen_customer,67574.0,15607.0,0.668655
seen_customer,284349.0,48120.0,0.685041


In [ ]:
promo_run = run_experiment(
    "lgb_with_promo_history",
    [
        "--cutoff", "2024-07-01",
        "--iterations", "2000",
        "--include-promo",
    ],
)

promo_metrics = json.loads(
    (promo_run / "metrics.json").read_text()
)
print("Promotion AUC:", promo_metrics["auc"])
display(pd.DataFrame(promo_metrics["monthly_auc"]).T)

Running: /usr/bin/python3 -m optiml.train --prepared-dir /content/optiml_prepared/baseline-v1 --output-dir /content/optiml_runs/jay/20261013T124210Z_lgb_with_promo_history_a54e1201 --model lightgbm --device CPU --threads 2 --cutoff 2024-07-01 --iterations 2000 --include-promo
Backup: /content/drive/MyDrive/OptimlRuns/jay/20261013T124210Z_lgb_with_promo_history_a54e1201
Promotion AUC: 0.6641843178656164


,rows,positives,auc
2024-07,59589.0,10757.0,0.697593
2024-08,58214.0,10560.0,0.682601
2024-09,55090.0,9937.0,0.659342
2024-10,61345.0,11256.0,0.654879
2024-11,58821.0,10517.0,0.651599
2024-12,58850.0,10698.0,0.637260
2025-01,14.0,2.0,0.333333


In [ ]:
recent_run = run_experiment(
    "lgb_no_promo_last12m",
    [
        "--cutoff", "2024-07-01",
        "--iterations", "2000",
        "--train-months", "12",
    ],
)

recent_metrics = json.loads(
    (recent_run / "metrics.json").read_text()
)
print("Recent-history AUC:", recent_metrics["auc"])
display(pd.DataFrame(recent_metrics["monthly_auc"]).T)

Running: /usr/bin/python3 -m optiml.train --prepared-dir /content/optiml_prepared/baseline-v1 --output-dir /content/optiml_runs/jay/20261013T124756Z_lgb_no_promo_last12m_622571fa --model lightgbm --device CPU --threads 2 --cutoff 2024-07-01 --iterations 2000 --train-months 12
Backup: /content/drive/MyDrive/OptimlRuns/jay/20261013T124756Z_lgb_no_promo_last12m_622571fa
Recent-history AUC: 0.684348368642165


,rows,positives,auc
2024-07,59589.0,10757.0,0.690856
2024-08,58214.0,10560.0,0.688626
2024-09,55090.0,9937.0,0.668045
2024-10,61345.0,11256.0,0.678350
2024-11,58821.0,10517.0,0.690012
2024-12,58850.0,10698.0,0.690131
2025-01,14.0,2.0,0.708333


In [ ]:
from argparse import Namespace
from datetime import datetime, timezone
from pathlib import Path
import gc
import json
import shutil
import uuid
import pandas as pd
from optiml.train import fit_model, model_inputs, write_submission

BASELINE_RUN = Path(full_run)
config = json.loads((BASELINE_RUN / "config.json").read_text())
metrics = json.loads((BASELINE_RUN / "metrics.json").read_text())

if not (BASELINE_RUN / "COMPLETE.json").is_file():
    raise RuntimeError("Baseline run is not complete")
if (
    config["model"] != "lightgbm"
    or config["include_promo"]
    or config["no_history"]
    or config["train_months"] is not None
    or config["sample_order_fraction"] != 1
):
    raise ValueError("Select the full-history, no-promotion, full-data LightGBM baseline run")

best_iterations = int(metrics["best_iterations"])
if best_iterations < 1:
    raise ValueError("Baseline metrics contain invalid best_iterations")

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_first_submission_" + uuid.uuid4().hex[:8]
local_run = Path("/content/optiml_runs") / OWNER / run_id
backup_run = DRIVE_RUNS / run_id
final_dir = local_run / "final"
final_dir.mkdir(parents=True, exist_ok=False)

try:
    train_frame = pd.read_parquet(PREPARED / "train.parquet")
    test_frame = pd.read_parquet(PREPARED / "test.parquet")
    test_ids = test_frame.order_line_id.copy()

    x, z, schema = model_inputs(
        train_frame, test_frame,
        include_promo=False, history=True, family="lightgbm"
    )
    y = train_frame.returned.to_numpy(copy=True)
    del train_frame, test_frame
    gc.collect()

    model_args = Namespace(**config)
    model, _ = fit_model(
        x, y, None, None, schema, model_args, final_dir, best_iterations
    )
    predictions = model.predict_proba(z)[:, 1]
    sample = pd.read_parquet(PREPARED / "sample_submission.parquet")
    submission_path = local_run / "submission.csv"
    write_submission(sample, test_ids, predictions, submission_path)

    (local_run / "first_submission.json").write_text(json.dumps({
        "baseline_run": str(BASELINE_RUN),
        "baseline_auc": metrics["auc"],
        "iterations": best_iterations,
        "test_rows": len(test_ids),
        "submission": str(submission_path),
    }, indent=2))

    print("Submission:", submission_path)
    print("Rows:", len(test_ids), "| baseline AUC:", metrics["auc"],
          "| iterations:", best_iterations)
finally:
    if local_run.exists():
        shutil.copytree(local_run, backup_run, dirs_exist_ok=True)
        print("Drive backup:", backup_run)

Submission: /content/optiml_runs/jay/20261013T130209Z_first_submission_b44f570c/submission.csv
Rows: 479995 | baseline AUC: 0.6864069862149578 | iterations: 137
Drive backup: /content/drive/MyDrive/OptimlRuns/jay/20261013T130209Z_first_submission_b44f570c


In [ ]:
from google.colab import files
files.download(str(backup_run / "submission.csv"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>